# Module 6: Apriori Frequent Itemsets, Null-Invariant Evaluation & Bloom Filters

**Z3003 Data Mining: Scaling to Large Data (BS Core) · IIT Madras Zanzibar**

Apriori pruning, Lift/Kulczynski/IR evaluation on Zanzibar agro-baskets, and Bloom filter calibration.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/dm-scale/labs/DM06_Frequent_Patterns_Apriori_Bloom.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every method is self-contained and verified with assertions.

## 1. Apriori Pruning & Null-Invariant Rule Evaluation

Apriori prunes any candidate $k$-itemset with an infrequent $(k-1)$-subset. Afterward, Lift and null-invariant Kulczynski + Imbalance Ratio filter out popular decoy items such as `hoe`.

In [1]:
from itertools import combinations
import math


def apriori(transactions, min_support):
    n = len(transactions)
    min_cnt = math.ceil(min_support * n)
    c1 = {}
    for b in transactions:
        for it in b:
            c1[it] = c1.get(it, 0) + 1
    freq = {frozenset([it]): c for it, c in c1.items() if c >= min_cnt}
    cur = sorted(freq, key=sorted)
    k = 2
    while cur:
        prev = set(cur)
        cands = {
            cur[i] | cur[j]
            for i in range(len(cur))
            for j in range(i + 1, len(cur))
            if len(cur[i] | cur[j]) == k and all(frozenset(s) in prev for s in combinations(cur[i] | cur[j], k - 1))
        }
        if not cands:
            break
        cc = {c: sum(1 for b in transactions if c.issubset(b)) for c in cands}
        nxt = [c for c, cnt in cc.items() if cnt >= min_cnt]
        for c in nxt:
            freq[c] = cc[c]
        cur = sorted(nxt, key=sorted)
        k += 1
    return freq


baskets = (
    [{"rice_seed", "urea", "hoe"} for _ in range(30)]
    + [{"hoe"} for _ in range(60)]
    + [{"rice_seed", "urea"} for _ in range(10)]
)
freq = apriori(baskets, min_support=0.25)
for itemset, cnt in sorted(freq.items(), key=lambda kv: (-kv[1], sorted(kv[0]))):
    print(f"{sorted(itemset)}: count={cnt}")

# Evaluate rice_seed -> urea vs rice_seed -> hoe
n = len(baskets)
c_rs = freq[frozenset({"rice_seed"})]
for target in ("urea", "hoe"):
    both = freq[frozenset({"rice_seed", target})]
    ct = freq[frozenset({target})]
    conf = both / c_rs
    lift = conf / (ct / n)
    kulc = 0.5 * (both / c_rs + both / ct)
    ir = abs(c_rs - ct) / (c_rs + ct - both)
    print(f"rice_seed -> {target:4s} | conf={conf:.2f}, lift={lift:.2f}, Kulc={kulc:.2f}, IR={ir:.2f}")


['hoe']: count=90
['rice_seed']: count=40
['rice_seed', 'urea']: count=40
['urea']: count=40
['hoe', 'rice_seed']: count=30
['hoe', 'rice_seed', 'urea']: count=30
['hoe', 'urea']: count=30
rice_seed -> urea | conf=1.00, lift=2.50, Kulc=1.00, IR=0.00
rice_seed -> hoe  | conf=0.75, lift=0.83, Kulc=0.54, IR=0.50


## 2. Bloom Filter Membership & Theoretical False-Positive Curve

A Bloom filter guarantees zero false negatives in $O(k)$ time per item, with false-positive rate $(1 - e^{-kn/m})^k$.

In [2]:
import hashlib


class BloomFilter:
    def __init__(self, m_bits, k_hashes):
        self.m, self.k = m_bits, k_hashes
        self.bits = bytearray(m_bits)

    def _pos(self, item):
        return [
            int.from_bytes(hashlib.blake2b(f"{s}:{item}".encode(), digest_size=8).digest(), "big") % self.m
            for s in range(self.k)
        ]

    def add(self, item):
        for p in self._pos(item):
            self.bits[p] = 1

    def contains(self, item):
        return all(self.bits[p] == 1 for p in self._pos(item))


m, n_items, k = 4096, 400, 5
bf = BloomFilter(m, k)
for i in range(n_items):
    bf.add(f"basket_{i}")
assert all(bf.contains(f"basket_{i}") for i in range(n_items))
fp_emp = sum(1 for i in range(n_items, n_items + 2000) if bf.contains(f"basket_{i}")) / 2000
fp_thy = (1 - math.exp(-k * n_items / m)) ** k
print(f"Bloom empirical FP = {fp_emp:.4f} | theoretical = {fp_thy:.4f}")
assert fp_emp < 0.05


Bloom empirical FP = 0.0055 | theoretical = 0.0086
